# Taller: Inferencia Estadística en Bioinformática
**Bioinformática para Ingenieros**  
*Msc. Juan Sebastian Malagón Torres & Eliana Zuluaga Aponte*  
*Pontificia Universidad Javeriana*

---

## Objetivos del Taller
1. Comprender la diferencia práctica entre **pruebas paramétricas y no paramétricas** en el análisis de datos biológicos.
2. Evaluar formalmente los **supuestos estadísticos** (normalidad y homocedasticidad) previos a una comparación de medias.
3. Aplicar e interpretar **ANOVA de una vía** y comprender cuándo y por qué recurrir a la alternativa no paramétrica de **Kruskal-Wallis**.
4. Contrastar los coeficientes de correlación de **Pearson y Spearman**, identificando el impacto de valores atípicos (*outliers*) y relaciones no lineales.
5. Emplear la prueba de **Chi-cuadrado ($\chi^2$) de independencia** para analizar la asociación entre variantes genéticas (SNPs) y fenotipos clínicos mediante tablas de contingencia.

> **Regla de oro de esta práctica:** Utilizaremos exclusivamente las librerías científicas fundamentales de Python: `numpy`, `pandas` y `scipy`. No se requieren paquetes externos adicionales.


## 0. Configuración del Entorno y Librerías

Importamos los tres pilares de computación científica y estadística en Python:
- `numpy`: Manipulación de arreglos numéricos y generación de números aleatorios.
- `pandas`: Manejo de estructuras de datos tabulares (`DataFrame`, `Series`).
- `scipy.stats`: Módulo especializado en distribuciones y pruebas de hipótesis estadísticas.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Fijamos una semilla aleatoria para garantizar reproducibilidad en las simulaciones
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("¡Librerías importadas exitosamente!")
print(f"NumPy versión: {np.__version__}")
print(f"Pandas versión: {pd.__version__}")
import scipy
print(f"SciPy versión: {scipy.__version__}")


---
## Ejercicio 1: Comparación de Grupos Paramétrica — ANOVA de una vía (One-way ANOVA)

### Contexto Biológico
En un estudio de transcriptómica sobre cáncer de mama, se cuantificaron los niveles de expresión normalizada ($\log_2(\text{TPM})$) del gen supresor de tumores ***BRCA1*** en biopsias tumorales de pacientes clasificadas en tres subtipos moleculares conocidos:
- **Luminal A** ($n = 35$)
- **Luminal B** ($n = 35$)
- **Basal-like** ($n = 35$)

El objetivo es determinar si existen diferencias significativas en la expresión media de *BRCA1* entre los tres subtipos tumorales.

### Fundamento Teórico (Ver Slides 9 y 10)
El análisis de varianza (ANOVA) compara las medias entre $k \ge 3$ grupos independientes evaluando la razón entre la variabilidad *entre grupos* y la variabilidad *intra-grupo*:

$$F = \frac{MSB}{MSW} = \frac{SSB / (k - 1)}{SSW / (N - k)}$$

Donde:
- $H_0: \mu_{\text{LumA}} = \mu_{\text{LumB}} = \mu_{\text{Basal}}$ (todas las medias poblacionales son iguales).
- $H_1$: Al menos una media poblacional difiere de las demás.
- **Supuestos obligatorios:**
  1. **Normalidad:** Las observaciones de cada grupo siguen una distribución normal.
  2. **Homocedasticidad:** Las varianzas de los grupos son homogéneas (iguales).
  3. **Independencia:** Las muestras provienen de pacientes independientes.


### 1.1 Simulación de los Datos de Expresión
Generamos datos de expresión para los tres subtipos que cumplen con los supuestos paramétricos.


In [ ]:
# Parámetros biológicos simulados
n_samples = 35

# Expresión log2(TPM) con medias biológicamente diferenciadas pero varianza común sigma=0.8
expr_lumA = np.random.normal(loc=8.2, scale=0.8, size=n_samples)
expr_lumB = np.random.normal(loc=7.6, scale=0.85, size=n_samples)
expr_basal = np.random.normal(loc=6.1, scale=0.82, size=n_samples)

# Construcción del DataFrame
df_anova = pd.DataFrame({
    'Paciente_ID': [f"BRCA_{i+1:03d}" for i in range(n_samples * 3)],
    'Subtipo': ['Luminal A'] * n_samples + ['Luminal B'] * n_samples + ['Basal-like'] * n_samples,
    'BRCA1_log2TPM': np.concatenate([expr_lumA, expr_lumB, expr_basal])
})

# Resumen descriptivo por subtipo
print("=== Resumen Estadístico por Subtipo Tumoral ===")
resumen_anova = df_anova.groupby('Subtipo')['BRCA1_log2TPM'].agg(['count', 'mean', 'std', 'median', lambda x: stats.iqr(x)])
resumen_anova.columns = ['N', 'Media', 'Desv_Std', 'Mediana', 'IQR']
print(resumen_anova.round(3))


### 1.2 Verificación de Supuestos: Normalidad y Homocedasticidad

Antes de realizar la prueba ANOVA, debemos verificar si los datos cumplen las condiciones matemáticas requeridas:

1. **Prueba de Normalidad de Shapiro-Wilk (`stats.shapiro`):**
   - $H_0$: Los datos provienen de una distribución normal.
   - $H_1$: Los datos NO provienen de una distribución normal.
   - *Criterio de decisión:* Si $p > 0.05$, **no se rechaza** $H_0$ (se asume normalidad).

2. **Prueba de Homocedasticidad de Levene (`stats.levene`):**
   - $H_0$: Las varianzas de todos los grupos son iguales.
   - $H_1$: Al menos un grupo presenta una varianza diferente.
   - *Criterio de decisión:* Si $p > 0.05$, **no se rechaza** $H_0$ (se asume homogeneidad de varianzas).


In [ ]:
print("--- 1. Evaluación de Normalidad (Shapiro-Wilk) ---")
for subtipo, grupo in df_anova.groupby('Subtipo'):
    stat_shapiro, p_shapiro = stats.shapiro(grupo['BRCA1_log2TPM'])
    es_normal = "Normal (p > 0.05)" if p_shapiro > 0.05 else "NO Normal (p <= 0.05)"
    print(f"Subtipo {subtipo:12s}: W = {stat_shapiro:.4f}, p-valor = {p_shapiro:.4f} -> {es_normal}")

print("\n--- 2. Evaluación de Homocedasticidad (Levene) ---")
grupo_lumA = df_anova[df_anova['Subtipo'] == 'Luminal A']['BRCA1_log2TPM']
grupo_lumB = df_anova[df_anova['Subtipo'] == 'Luminal B']['BRCA1_log2TPM']
grupo_basal = df_anova[df_anova['Subtipo'] == 'Basal-like']['BRCA1_log2TPM']

stat_levene, p_levene = stats.levene(grupo_lumA, grupo_lumB, grupo_basal)
es_homocedastico = "Varianzas Homogéneas (p > 0.05)" if p_levene > 0.05 else "Heterocedasticidad (p <= 0.05)"
print(f"Estadístico de Levene: {stat_levene:.4f}, p-valor = {p_levene:.4f} -> {es_homocedastico}")


### 1.3 Aplicación de ANOVA de una vía (`stats.f_oneway`)

Dado que se cumplen ambos supuestos, procedemos con la prueba ANOVA paramétrica.


In [ ]:
# Ejecución del ANOVA
f_stat, p_val_anova = stats.f_oneway(grupo_lumA, grupo_lumB, grupo_basal)

print("=== RESULTADOS ANOVA DE UNA VÍA ===")
print(f"Estadístico F: {f_stat:.4f}")
print(f"p-valor:       {p_val_anova:.4e}")

alpha = 0.05
if p_val_anova < alpha:
    print(f"\nConclusión: Se rechaza H0 (p < {alpha}). Existen diferencias estadísticamente significativas en la expresión de BRCA1 entre subtipos.")
else:
    print(f"\nConclusión: No se rechaza H0 (p >= {alpha}). No hay evidencia suficiente de diferencias entre los grupos.")


### ✍️ Preguntas de Análisis — Ejercicio 1
1. **Interpretación del estadístico $F$:** De acuerdo con la fórmula $F = MSB / MSW$ vista en clase, ¿qué significa biológicamente e intuitivamente que el valor de $F$ sea grande en comparación con $1$?
2. **Limitación de ANOVA:** La prueba nos indica si *al menos un grupo* difiere, pero no cuál de ellos es diferente de los demás. ¿Qué tipo de prueba posterior (*post-hoc*) se suele aplicar en bioinformática tras rechazar la hipótesis nula en ANOVA?
3. **Consecuencia biológica:** Si un investigador hubiera saltado la verificación de normalidad y homocedasticidad, ¿por qué en este caso particular el resultado de ANOVA sigue siendo válido?


---
## Ejercicio 2: Ruptura de Supuestos — De ANOVA a Kruskal-Wallis

### Contexto Biológico
En un estudio de metagenómica del microbioma intestinal humano, se cuantificó la abundancia relativa (conteos por millón de lecturas) de la bacteria oportunista ***Bacteroides fragilis*** en tres cohortes de individuos sometidos a diferentes dietas:
- **Dieta Estándar / Control** ($n = 40$)
- **Dieta Alta en Fibra** ($n = 40$)
- **Dieta Alta en Grasas / Ultraprocesados** ($n = 40$)

Los datos de abundancia taxonómica en metagenómica y secuenciación masiva (16S rRNA / RNA-seq crudo) presentan típicamente:
1. Una alta proporción de ceros (ausencia de lecturas en varias muestras).
2. Marcada asimetría positiva (la mayoría de pacientes tienen abundancias bajas o nulas, mientras unos pocos tienen picos extremos).
3. Varianza heterogénea entre grupos.

### Fundamento Teórico (Ver Slide 11)
La prueba de **Kruskal-Wallis** es la extensión no paramétrica del ANOVA de una vía para $k \ge 3$ grupos independientes:
- **No asume distribución normal**; en su lugar, transforma todos los valores continuos en **rangos** ($1$ a $N$).
- Evalúa si las distribuciones de los rangos (medianas) provienen de la misma población:
  - $H_0$: Todas las poblaciones tienen la misma distribución de rangos (medianas iguales).
  - $H_1$: Al menos una población tiene una distribución de rangos sistemáticamente diferente.

El estadístico $H$ se calcula como:

$$H = \left[ \frac{12}{N(N+1)} \right] \sum_{i=1}^k \frac{R_i^2}{n_i} - 3(N+1)$$

Bajo $H_0$, $H$ sigue asintóticamente una distribución Chi-cuadrado con $k - 1$ grados de libertad.


### 2.1 Simulación de Abundancias de Microbioma (Datos Asimétricos y con Ceros)


In [ ]:
n_micro = 40

# Dieta Control: distribución log-normal con abundancias bajas
ctrl = np.random.lognormal(mean=1.2, sigma=1.0, size=n_micro)
ctrl[np.random.choice(n_micro, 10, replace=False)] = 0.0 # Ceros técnicos/biológicos

# Dieta Alta en Fibra: supresión de Bacteroides fragilis (mayoría valores muy bajos o cero)
fibra = np.random.lognormal(mean=0.4, sigma=0.8, size=n_micro)
fibra[np.random.choice(n_micro, 18, replace=False)] = 0.0

# Dieta Alta en Grasa: sobrecrecimiento con amplia dispersión y outliers biológicos
grasa = np.random.lognormal(mean=2.8, sigma=1.5, size=n_micro)

df_micro = pd.DataFrame({
    'Muestra_ID': [f"SMP_{i+1:03d}" for i in range(n_micro * 3)],
    'Dieta': ['Control'] * n_micro + ['Alta Fibra'] * n_micro + ['Alta Grasa'] * n_micro,
    'Abundancia_B_fragilis': np.concatenate([ctrl, fibra, grasa])
})

print("=== Estadísticas Descriptivas de Abundancia ===")
resumen_micro = df_micro.groupby('Dieta')['Abundancia_B_fragilis'].agg(
    ['count', 'mean', 'std', 'median', lambda x: stats.iqr(x), lambda x: (x == 0).sum()]
)
resumen_micro.columns = ['N', 'Media', 'Desv_Std', 'Mediana', 'IQR', 'Conteo_Ceros']
print(resumen_micro.round(3))


### 2.2 Verificación de Supuestos: Demostrando el Fallo de ANOVA
Evaluemos formalmente la normalidad y la homocedasticidad sobre estos datos de microbioma.


In [ ]:
print("--- 1. Prueba de Shapiro-Wilk en Datos de Microbioma ---")
for dieta, grupo in df_micro.groupby('Dieta'):
    stat_w, p_w = stats.shapiro(grupo['Abundancia_B_fragilis'])
    cumple = "NORMAL (p > 0.05)" if p_w > 0.05 else "NO NORMAL (p <= 0.05) [VIOLACIÓN]"
    print(f"Grupo {dieta:12s}: W = {stat_w:.4f}, p-valor = {p_w:.4e} -> {cumple}")

print("\n--- 2. Prueba de Levene (Homocedasticidad) ---")
g_ctrl = df_micro[df_micro['Dieta'] == 'Control']['Abundancia_B_fragilis']
g_fibra = df_micro[df_micro['Dieta'] == 'Alta Fibra']['Abundancia_B_fragilis']
g_grasa = df_micro[df_micro['Dieta'] == 'Alta Grasa']['Abundancia_B_fragilis']

stat_lev, p_lev = stats.levene(g_ctrl, g_fibra, g_grasa)
cumple_lev = "HOMOCEDÁSTICO (p > 0.05)" if p_lev > 0.05 else "HETEROCEDÁSTICO (p <= 0.05) [VIOLACIÓN]"
print(f"Levene W = {stat_lev:.4f}, p-valor = {p_lev:.4e} -> {cumple_lev}")


### 2.3 Comparación: ANOVA (Inadecuado) vs. Kruskal-Wallis (Adecuado)

Ejecutemos ambas pruebas para contrastar el comportamiento y la validez metodológica.


In [ ]:
# 1. ¿Qué pasaría si aplicamos ANOVA de forma ciega?
f_val, p_val_f = stats.f_oneway(g_ctrl, g_fibra, g_grasa)

# 2. Aplicación correcta: Kruskal-Wallis (scipy.stats.kruskal)
h_val, p_val_kw = stats.kruskal(g_ctrl, g_fibra, g_grasa)

print("=== COMPARACIÓN METODOLÓGICA ===")
print(f"ANOVA (Paramétrico):       F = {f_val:.4f}, p-valor = {p_val_f:.4e}  <- [SUPUESTOS VIOLADOS]")
print(f"Kruskal-Wallis (No param): H = {h_val:.4f}, p-valor = {p_val_kw:.4e} <- [PRUEBA VÁLIDA]")

# Verificación de significancia con Kruskal-Wallis
alpha = 0.05
if p_val_kw < alpha:
    print(f"\nConclusión Kruskal-Wallis: Se rechaza H0 (p < {alpha}). Las distribuciones de abundancia de B. fragilis difieren significativamente entre dietas.")
else:
    print(f"\nConclusión Kruskal-Wallis: No se rechaza H0 (p >= {alpha}). No hay evidencia de diferencias en las distribuciones.")


### ✍️ Preguntas de Análisis — Ejercicio 2
1. **Mediana vs. Media en datos sesgados:** Observa la tabla de resumen estadístico del punto 2.1 para el grupo `Alta Grasa`. ¿Por qué la media ($~\bar{x}$) es mucho mayor que la mediana? ¿Cuál de las dos medidas representa mejor al "individuo típico" de este grupo?
2. **Transformación de rangos:** Explica por qué transformar los datos continuos a sus rangos ordinales ($1, 2, \dots, N$) hace que Kruskal-Wallis sea resistente a la presencia de ceros y a valores extremadamente altos (*outliers*).
3. **Toma de decisiones en bioinformática:** Menciona otros dos tipos de datos bioinformáticos (aparte de abundancia de microbioma) donde frecuentemente no se cumpla el supuesto de normalidad y deba optarse por pruebas no paramétricas.


---
## Ejercicio 3: Correlación Paramétrica vs. No Paramétrica — Pearson vs. Spearman

### Contexto Biológico
El análisis de correlación cuantifica la fuerza y dirección de la relación entre dos variables biológicas cuantitativas:
- **Escenario A (Relación Lineal Coexpresión Génica):** Se analiza la coexpresión de dos enzimas metabólicas clave (*GAPDH* y *PKM*) que pertenecen a la misma ruta glicolítica en $n = 50$ muestras celulares. Se espera una relación biológica continua, proporcional y sin perturbaciones atípicas.
- **Escenario B (Relación Monotónica No Lineal y Outliers Técnicos):** Se evalúa la relación entre la profundidad de secuenciación (*Coverage / Depth*, lecturas en millones) y el número de variantes genéticas raras detectadas en $n = 50$ librerías. Al aumentar la cobertura, el número de variantes nuevas detectadas experimenta una saturación (meseta logarítmica), y adicionalmente 2 muestras sufrieron un error técnico de contaminación (generando valores atípicos extremos).

### Fundamento Teórico (Ver Slides 13 y 14)

| Característica | Correlación de Pearson ($r$) | Correlación de Spearman ($\rho$) |
| :--- | :--- | :--- |
| **Naturaleza** | Paramétrica | No paramétrica (basada en rangos) |
| **Tipo de relación** | Estrictamente **lineal** | **Monotónica** (creciente o decreciente, lineal o no lineal) |
| **Supuestos** | Distribución normal bivariada, sin valores atípicos | No requiere normalidad ni relación lineal |
| **Sensibilidad a outliers**| **Muy alta** (un solo outlier puede distorsionar $r$) | **Baja** (muy robusta a outliers) |
| **Fórmula del estadístico**| $r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \sum(y_i - \bar{y})^2}}$ | $\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$ (donde $d_i = \text{rango}(x_i) - \text{rango}(y_i)$) |


### 3.1 Simulación de Escenarios A y B


In [ ]:
n_corr = 50

# --- ESCENARIO A: Coexpresión Lineal (GAPDH vs PKM) ---
gapdh = np.random.normal(loc=10.0, scale=1.5, size=n_corr)
# Relación lineal: PKM = 0.8 * GAPDH + ruido normal
pkm = 0.8 * gapdh + np.random.normal(loc=2.0, scale=0.6, size=n_corr)

df_lineal = pd.DataFrame({'GAPDH_expr': gapdh, 'PKM_expr': pkm})

# --- ESCENARIO B: Saturación Monotónica con Outliers Técnicos ---
# Cobertura de secuenciación (Depth en millones de lecturas)
cobertura = np.sort(np.random.uniform(5.0, 100.0, size=n_corr))
# Curva de saturación logarítmica monotónica
variantes_base = 150 + 80 * np.log(cobertura) + np.random.normal(0, 10, size=n_corr)

# Introducimos 2 outliers extremos por error de artefacto técnico en las últimas dos muestras
variantes_con_outliers = variantes_base.copy()
variantes_con_outliers[-1] = 20.0   # Muestra con falla de librería (pérdida masiva de llamadas)
variantes_con_outliers[-2] = 900.0  # Muestra con artefacto de secuenciación (quimeras/duplicados)

df_saturacion = pd.DataFrame({
    'Cobertura_MReads': cobertura,
    'Variantes_Detectadas': variantes_con_outliers
})

print("Datos simulados para Escenario A y Escenario B exitosamente.")


### 3.2 Evaluación Estadística en Ambos Escenarios (`stats.pearsonr` vs. `stats.spearmanr`)


In [ ]:
# --- Escenario A: Coexpresión Lineal ---
r_pearson_A, p_pearson_A = stats.pearsonr(df_lineal['GAPDH_expr'], df_lineal['PKM_expr'])
rho_spearman_A, p_spearman_A = stats.spearmanr(df_lineal['GAPDH_expr'], df_lineal['PKM_expr'])

# --- Escenario B: Saturación con Outliers Técnicos ---
r_pearson_B, p_pearson_B = stats.pearsonr(df_saturacion['Cobertura_MReads'], df_saturacion['Variantes_Detectadas'])
rho_spearman_B, p_spearman_B = stats.spearmanr(df_saturacion['Cobertura_MReads'], df_saturacion['Variantes_Detectadas'])

# Presentación comparativa en un DataFrame
tabla_correlaciones = pd.DataFrame({
    'Escenario': [
        'A. Lineal (GAPDH vs PKM)', 
        'B. No Lineal + Outliers (Cobertura vs Variantes)'
    ],
    'Pearson_r': [r_pearson_A, r_pearson_B],
    'Pearson_p_valor': [p_pearson_A, p_pearson_B],
    'Spearman_rho': [rho_spearman_A, rho_spearman_B],
    'Spearman_p_valor': [p_spearman_A, p_spearman_B]
})

print("=== TABLA COMPARATIVA: PEARSON VS SPEARMAN ===")
print(tabla_correlaciones.to_string(index=False))


### ✍️ Preguntas de Análisis — Ejercicio 3
1. **Comportamiento en el Escenario A:** ¿Por qué en el Escenario A tanto el coeficiente de Pearson como el de Spearman arrojan valores muy similares y altamente significativos?
2. **Impacto de la no linealidad y outliers en el Escenario B:** Observa la discrepancia en el Escenario B entre $r$ de Pearson y $\rho$ de Spearman. ¿Qué coeficiente refleja mejor la tendencia biológica real y por qué Pearson se ve penalizado?
3. **Kendall Tau (Slide 14):** En las diapositivas también se introdujo el coeficiente $\tau$ de Kendall basado en pares concordantes y discordantes. ¿Bajo qué condiciones (`stats.kendalltau`) se preferiría Kendall sobre Spearman en el análisis bioinformático?


---
## Ejercicio 4: Análisis de Frecuencias e Independencia — Prueba Chi-cuadrado ($\chi^2$)

### Contexto Biológico
En un estudio de genómica clínica y farmacogenómica, se analizó un polimorfismo de un solo nucleótido (**SNP** bialélico $A/G$, identificador `rs1045642`) en $n = 180$ pacientes oncológicos tratados con una terapia dirigida.

A cada paciente se le determinó su:
1. **Genotipo del SNP:**
   - `A/A` (Homocigoto de referencia)
   - `A/G` (Heterocigoto)
   - `G/G` (Homocigoto alternativo)
2. **Respuesta Terapéutica (Fenotipo clínico):**
   - `Respondedor` (Remisión tumoral o estabilización de la enfermedad)
   - `No Respondedor` (Progresión tumoral / resistencia al fármaco)

El objetivo es determinar si existe una **asociación estadísticamente significativa** entre el genotipo del paciente y su probabilidad de responder al tratamiento.

### Fundamento Teórico (Ver Slides 15 y 16)
La prueba de Chi-cuadrado ($\chi^2$) de independencia evalúa si dos variables categóricas son independientes o si están asociadas:
- $H_0$: Las dos variables son **independientes** (la respuesta terapéutica no depende del genotipo).
- $H_1$: Las dos variables **NO son independientes** (existe una asociación entre el genotipo y la respuesta).

Para calcular el estadístico, primero se obtienen las frecuencias observadas ($O_i$) en una tabla de contingencia de dimensiones $r \times c$. Luego se calculan las frecuencias esperadas bajo el supuesto de independencia ($E_i$):

$$E_{ij} = \frac{\text{Total Fila}_i \times \text{Total Columna}_j}{N}$$

El estadístico de prueba es:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

Bajo $H_0$, este estadístico sigue una distribución $\chi^2$ con grados de libertad:

$$\text{df} = (r - 1) \times (c - 1)$$

> **Regla empírica obligatoria (Slide 16):** Todas las celdas de la tabla de frecuencias esperadas deben tener un valor mínimo de $E_i \ge 5$ para que la aproximación asintótica sea válida.


### 4.1 Simulación de la Cohorte de Pacientes


In [ ]:
n_pacientes = 180

# Simulación de genotipos según frecuencias de equilibrio poblacional aproximadas
genotipos_posibles = ['A/A', 'A/G', 'G/G']
p_genotipos = [0.45, 0.40, 0.15] # Frecuencias relativas

genotipos_simulados = np.random.choice(genotipos_posibles, size=n_pacientes, p=p_genotipos)

# Simulación de respuesta clínica ligada al alelo de riesgo 'G'
# Los pacientes con alelo 'G' presentan menor tasa de respuesta terapéutica
respuestas = []
for g in genotipos_simulados:
    if g == 'A/A':
        # 75% de probabilidad de responder favorablemente
        resp = np.random.choice(['Respondedor', 'No Respondedor'], p=[0.75, 0.25])
    elif g == 'A/G':
        # 45% de probabilidad de responder favorablemente
        resp = np.random.choice(['Respondedor', 'No Respondedor'], p=[0.45, 0.55])
    else: # 'G/G'
        # 15% de probabilidad de responder favorablemente
        resp = np.random.choice(['Respondedor', 'No Respondedor'], p=[0.15, 0.85])
    respuestas.append(resp)

df_gwas = pd.DataFrame({
    'Paciente_ID': [f"PAT_{i+1:03d}" for i in range(n_pacientes)],
    'Genotipo_rs1045642': genotipos_simulados,
    'Respuesta_Clinica': respuestas
})

print(f"Total pacientes simulados: {len(df_gwas)}")
print(df_gwas.head(10))


### 4.2 Construcción de la Tabla de Contingencia y Verificación de Frecuencias Esperadas

Utilizamos `pd.crosstab` para construir la matriz de frecuencias observadas ($O_{ij}$).


In [ ]:
# 1. Tabla de contingencia observada (O)
tabla_observada = pd.crosstab(
    df_gwas['Genotipo_rs1045642'], 
    df_gwas['Respuesta_Clinica'], 
    margins=True, 
    margins_name="Total"
)

print("=== TABLA DE CONTINGENCIA: FRECUENCIAS OBSERVADAS (O) ===")
print(tabla_observada)


### 4.3 Ejecución de la Prueba Chi-cuadrado de Independencia (`stats.chi2_contingency`)

La función `scipy.stats.chi2_contingency` recibe la tabla de frecuencias observadas (sin los márgenes de totales) y calcula automáticamente:
1. El estadístico $\chi^2$.
2. El p-valor correspondiente.
3. Los grados de libertad ($\text{df}$).
4. La matriz completa de frecuencias esperadas ($E_{ij}$).


In [ ]:
# Matriz observada sin márgenes de totales para scipy
matriz_obs = pd.crosstab(df_gwas['Genotipo_rs1045642'], df_gwas['Respuesta_Clinica'])

# Ejecución de la prueba Chi-cuadrado
chi2_stat, p_val_chi2, dof, esperados = stats.chi2_contingency(matriz_obs)

# Conversión de la matriz de esperados a un DataFrame legible
df_esperados = pd.DataFrame(
    esperados, 
    index=matriz_obs.index, 
    columns=matriz_obs.columns
)

print("=== FRECUENCIAS ESPERADAS BAJO H0 (E_ij) ===")
print(df_esperados.round(2))

# Verificación de la regla empírica (E_ij >= 5)
min_esperado = esperados.min()
print(f"\nFrecuencia esperada mínima en la tabla: {min_esperado:.2f}")
if min_esperado >= 5.0:
    print("-> CUMPLE la regla empírica (todas las celdas E_ij >= 5). La prueba de Chi-cuadrado es confiable.")
else:
    print("-> ALERTA: Hay celdas con E_ij < 5. En casos reales debería usarse la Prueba Exacta de Fisher.")

print("\n=== RESULTADOS DE LA PRUEBA CHI-CUADRADO ===")
print(f"Estadístico Chi2:     {chi2_stat:.4f}")
print(f"Grados de libertad:   {dof}")
print(f"p-valor:              {p_val_chi2:.4e}")

alpha = 0.05
if p_val_chi2 < alpha:
    print(f"\nConclusión: Se rechaza H0 (p < {alpha}). El genotipo del SNP rs1045642 y la respuesta clínica NO son independientes.")
else:
    print(f"\nConclusión: No se rechaza H0 (p >= {alpha}). No existe asociación estadísticamente significativa.")


### ✍️ Preguntas de Análisis — Ejercicio 4
1. **Grados de libertad:** Demuestra formalmente el cálculo de los grados de libertad para esta tabla de $3 \times 2$ usando la fórmula $\text{df} = (r - 1)(c - 1)$.
2. **Comparación Observados vs. Esperados:** Compara la celda `A/A - Respondedor` entre la tabla observada y la esperada. ¿Qué observas y qué sugiere clínicamente respecto al alelo de referencia `A`?
3. **¿Qué hacer si no se cumple $E_{ij} \ge 5$?** Si hubiésemos analizado una cohorte pequeña ($n = 15$) y la frecuencia esperada de alguna celda fuera menor a 5, ¿cuál es la prueba estadística exacta recomendada en bioinformática?


---
## 5. Tabla Resumen: Guía Rápida para Selección de Pruebas Estadísticas en Bioinformática

Para consolidar tu aprendizaje, guarda esta matriz de decisión para tus futuros análisis bioinformáticos:

| Pregunta de Investigación | Tipo de Variables | Supuesto de Normalidad | Prueba Paramétrica Recomendada | Alternativa No Paramétrica | Función en SciPy |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Comparar 2 grupos independientes** | Numérica continua vs. Categórica (2 niveles) | Cumple | *t-test* para muestras independientes | Mann-Whitney $U$ test | `stats.ttest_ind` / `stats.mannwhitneyu` |
| **Comparar $\ge 3$ grupos independientes** | Numérica continua vs. Categórica ($\ge 3$ niveles) | Cumple | **ANOVA de una vía** | **Kruskal-Wallis** | `stats.f_oneway` / `stats.kruskal` |
| **Relación entre 2 variables cuantitativas** | 2 variables numéricas continuas | Cumple (lineal y bivariada) | **Correlación de Pearson ($r$)** | **Correlación de Spearman ($\rho$) / Kendall ($\tau$)** | `stats.pearsonr` / `stats.spearmanr` / `stats.kendalltau` |
| **Asociación entre variables categóricas** | 2 variables cualitativas (tablas de contingencia) | N/A (frecuencias esperadas $\ge 5$) | N/A | **Prueba $\chi^2$ de independencia** (o Exacta de Fisher) | `stats.chi2_contingency` / `stats.fisher_exact` |

---
*Fin del Taller de Inferencia Estadística — Bioinformática para Ingenieros*
